<div style="background:linear-gradient(135deg,#b71c1c,#c62828,#d32f2f);
padding:36px 28px;border-radius:14px;color:#fff;text-align:center">
<h1 style="margin:0;font-size:2.1em">🏭 Caso Industrial 2 — Sector Olefinas</h1>
<h2 style="margin:10px 0 4px;font-weight:300;opacity:.94">
Craqueo Térmico de Etano a Etileno (Steam Cracking)</h2>
<p style="margin:6px 0 0;opacity:.85">
Reactores Batch · CSTR · PFR — Diseño Comparativo</p>
<p style="margin:4px 0 0;opacity:.75;font-size:.88em;font-style:italic">
740484 Diseño de Reactores · Universidad del Valle · Prof. José Antonio Lara Ramos</p>
</div>

---

## Contexto industrial

El **etileno** (C₂H₄) es el producto petroquímico de mayor volumen en el mundo
(~200 Mt/año). Es la materia prima de:

| Derivado | Aplicación | Vol. global |
|----------|-----------|-------------|
| Polietileno (HDPE, LDPE, LLDPE) | Empaques, tuberías, películas | ~100 Mt/año |
| Óxido de etileno → Etilenglicol | PET, anticongelantes | ~30 Mt/año |
| Cloruro de vinilo (PVC) | Construcción, tuberías | ~50 Mt/año |
| Estireno (via EB) | PS, ABS, SBR | ~30 Mt/año |
| Acetaldehído, ácido acético | Solventes, industria química | ~5 Mt/año |

**Proceso industrial:** Steam Cracking — reacción radicalaria a 750-900°C,
tiempo de residencia 0.1-0.5 s, con vapor de agua como diluyente.

---

## La reacción principal (modelo simplificado — 1er orden)

$$\underbrace{C_2H_6}_{\text{etano (A)}} \;\longrightarrow\;
\underbrace{C_2H_4}_{\text{etileno (E, B)}} + H_2$$

- **Endotérmica**: $\Delta H_{rxn}^\circ = +137\;\text{kJ/mol}$
- **Irreversible** a las temperaturas de operación ($K_{eq}$ enorme a 800°C)
- **1er orden**: $-r_A = k\,C_A$  con cinética de Arrhenius

### Datos cinéticos — Arrhenius

$$k(T) = k_0\,\exp\!\left(-\frac{E_a}{RT}\right)$$

| Parámetro | Valor | Unidades |
|-----------|-------|----------|
| $k_0$ (factor pre-exponencial) | $4.65 \times 10^{13}$ | s⁻¹ |
| $E_a$ (energía de activación) | $272\;\text{kJ/mol}$ | kJ/mol |
| $T$ operación | 800 | °C = 1073 K |
| $k$ a 800°C | ≈ 3.5 | s⁻¹ |

### Condiciones de diseño

| Variable | Valor | Unidades |
|----------|-------|----------|
| Temperatura | 800 | °C = 1073 K |
| Presión | 2.0 | bar |
| $C_{A0}$ (etano) | 0.025 | mol/dm³ |
| Caudal $v_0$ | 2000 | dm³/s |
| $F_{A0}$ | 50 | mol/s |
| Conversión deseada | 0.60 | — |

> **Nota:** El steam cracking opera con tiempos de residencia muy cortos
> (0.1-0.5 s) porque el etileno formado puede reaccionar secundariamente
> a acetileno, benceno y coque. El PFR tubular de alta temperatura es
> el único reactor viable industrialmente.

---

## Objetivos del caso

1. Calcular el **tiempo** en reactor **Batch** (equivalente al tiempo espacial del PFR).
2. Calcular el **volumen** del **CSTR** y demostrar su **inviabilidad** operacional.
3. Calcular el **volumen** del **PFR** y verificar el tiempo de residencia.
4. Demostrar con el **Diagrama de Levenspiel** por qué el CSTR es impractical aquí.
5. Analizar la **sensibilidad** a la temperatura via Arrhenius.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
from scipy.integrate import odeint, cumulative_trapezoid
from scipy.optimize import brentq
from ipywidgets import interact, FloatSlider
import warnings; warnings.filterwarnings('ignore')

plt.rcParams.update({
    'font.family':'DejaVu Sans','font.size':11,'axes.grid':True,
    'grid.alpha':0.28,'figure.dpi':110,'axes.titlesize':12,
    'lines.linewidth':2.3,'axes.titleweight':'bold',
})

# ============================================================
# PARAMETROS — Craqueo Termico de Etano
# ============================================================
R_gas  = 8.314e-3   # kJ/(mol*K)
k0     = 4.65e13    # s^-1   (factor pre-exponencial)
Ea     = 272.0      # kJ/mol (energia de activacion)
T_op   = 1073.0     # K (800°C)
k_op   = k0*np.exp(-Ea/(R_gas*T_op))

CA0    = 0.025      # mol/dm^3
v0     = 2000.0     # dm^3/s
FA0    = CA0*v0     # mol/s
Xdes   = 0.60       # conversion deseada

print("=" * 58)
print("  CRAQUEO TERMICO DE ETANO A ETILENO")
print("=" * 58)
print(f"  k0     = {k0:.3e}  s^-1")
print(f"  Ea     = {Ea:.1f}  kJ/mol")
print(f"  T_op   = {T_op:.0f}  K ({T_op-273.15:.0f} C)")
print(f"  k(T)   = {k_op:.4f}  s^-1")
print(f"  C_A0   = {CA0}  mol/dm3")
print(f"  v0     = {v0:.0f}  dm3/s")
print(f"  F_A0   = {FA0:.1f}  mol/s")
print(f"  X_des  = {Xdes}")
print()
# Verificar
Da_check = k_op * (Xdes/(1-Xdes)) / k_op  # solo para 1er orden
tau_check = Xdes/(k_op*(1-Xdes))
print(f"  tau necesario (CSTR 1er orden) = {tau_check:.4f} s")
print(f"  tau necesario (PFR 1er orden)  = {-np.log(1-Xdes)/k_op:.4f} s")
print("=" * 58)


---
## Paso 1 — Análisis cinético con Arrhenius y Diagrama de Levenspiel

### Ecuación de velocidad

Para 1er orden irreversible:

$$-r_A = k(T)\,C_A = k(T)\,C_{A0}(1-X)$$

### Sensibilidad a la temperatura (Arrhenius)

Con $E_a = 272\;\text{kJ/mol}$, la regla de los 10°C aplica de forma extrema:

$$\frac{k(T+10)}{k(T)} = \exp\!\left[\frac{E_a\cdot 10}{R\,T(T+10)}\right] \approx \exp\!\left[\frac{272 \times 10}{8.314\times10^{-3}\times 1073^2}\right] \approx 2.5$$

**Cada 10°C, la velocidad se multiplica por ~2.5x** — extremadamente sensible.
Esto explica por qué el control de temperatura en los hornos de craqueo es crítico
(±5°C puede duplicar o reducir a la mitad la conversión).

### Impacto en el diseño de reactor

Para cinética de 1er orden irreversible:
- **PFR**: $\tau_{PFR} = -\ln(1-X)/k$ — logarítmico, relativamente moderado
- **CSTR**: $\tau_{CSTR} = X/[k(1-X)]$ — hiperbólico, escala explosivamente hacia $X=1$
- **Razón**: $\tau_{CSTR}/\tau_{PFR} = X/[(1-X)(-\ln(1-X))]$

Para $X=0.60$: esta razón es $0.6/[0.4\times 0.916] = 1.63\times$

Para $X=0.90$: la razón sube a $3.91\times$ — el CSTR requiere casi 4 veces más volumen.


In [ ]:
# Paso 1: Cinética, Arrhenius y Levenspiel
X_arr = np.linspace(0.001, 0.999, 500)
CA_arr = CA0*(1 - X_arr)
rA_arr = k_op*CA_arr                # mol/(dm^3*s)

# Levenspiel
Lev_dm3 = FA0/rA_arr                # dm^3 (para V en dm^3)
Lev_s   = CA0/rA_arr                # s    (tiempo espacial)

# Punto de diseno
rA_des  = k_op*CA0*(1-Xdes)
Lev_des = FA0/rA_des

fig, axes = plt.subplots(1, 3, figsize=(17, 5.5))

# Panel 1: k(T) vs Temperatura (Arrhenius)
T_range = np.linspace(700, 1000, 300)
k_range = k0*np.exp(-Ea/(R_gas*T_range))
ax = axes[0]
ax.semilogy(T_range-273.15, k_range, '#C62828', lw=3)
ax.axvline(T_op-273.15, color='#1565C0', ls='--', lw=2.5,
           label=f'T_op={T_op-273.15:.0f}C: k={k_op:.3f} s^-1')
ax.scatter([T_op-273.15],[k_op], s=200, color='gold', zorder=9,
           edgecolors='black', lw=1.5, marker='*')
# Mostrar factor 10C
T_10 = T_op + 10
k_10 = k0*np.exp(-Ea/(R_gas*T_10))
ax.annotate(f'+10C -> k x{k_10/k_op:.2f}',
            xy=(T_10-273.15, k_10),
            xytext=(T_10-273.15+15, k_10*0.4),
            arrowprops=dict(arrowstyle='->', color='#2E7D32', lw=1.5),
            fontsize=9, color='#2E7D32', fontweight='bold')
ax.set_xlabel('Temperatura [C]')
ax.set_ylabel('k [s^-1]  (escala log)')
ax.set_title('Constante cinetica vs T\n(Arrhenius — Ea=272 kJ/mol)')
ax.legend(fontsize=9)

# Panel 2: Diagrama de Levenspiel
ax2 = axes[1]
mask = X_arr <= Xdes
ax2.plot(X_arr, Lev_s, '#1565C0', lw=3, label='C_A0/(-r_A) [s]')
# PFR (area bajo curva)
ax2.fill_between(X_arr[mask], 0, Lev_s[mask],
                 alpha=0.28, color='#2E7D32',
                 label=f'PFR: tau={-np.log(1-Xdes)/k_op:.4f} s')
# CSTR (rectangulo)
Ld = CA0/rA_des
ax2.fill_between([0,Xdes],[0,0],[Ld,Ld],
                 alpha=0.28, color='#C62828',
                 label=f'CSTR: tau={Xdes/(k_op*(1-Xdes)):.4f} s')
ax2.plot([0,Xdes,Xdes],[Ld,Ld,0],'#C62828',ls='--',lw=2)
ax2.set_xlabel('Conversion X_etano')
ax2.set_ylabel('C_A0/(-r_A) [s]')
ax2.set_title('Diagrama de Levenspiel\n'
              'Craqueo termico de etano')
ax2.legend(fontsize=9)
ax2.set_ylim(0, Ld*3.5)

# Panel 3: Comparacion tau_CSTR vs tau_PFR vs X
tau_PFR_arr  = -np.log(1-X_arr)/k_op
tau_CSTR_arr = X_arr/(k_op*(1-X_arr))
ratio_arr    = tau_CSTR_arr/tau_PFR_arr

ax3 = axes[2]
ax3.plot(X_arr, tau_PFR_arr,  '#2E7D32', lw=3, label='tau PFR [s]')
ax3.plot(X_arr, tau_CSTR_arr, '#C62828', lw=3, label='tau CSTR [s]')
ax3.axvline(Xdes, color='#1565C0', ls='--', lw=2.2,
            label=f'X_des={Xdes}')
ax3.set_xlabel('Conversion X_etano')
ax3.set_ylabel('Tiempo espacial tau [s]')
ax3.set_title('tau_PFR vs tau_CSTR\n'
              '(la diferencia crece con X)')
ax3.legend(fontsize=9); ax3.set_ylim(0, 3.0)

plt.suptitle('Caso 2 — Paso 1: Cinetica y Levenspiel\n'
             'Craqueo termico de etano a etileno (T=800C)',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print(f"  k(800C) = {k_op:.5f} s^-1")
print(f"  tau_PFR  = {-np.log(1-Xdes)/k_op:.5f} s  ({-np.log(1-Xdes)/k_op*1000:.2f} ms)")
print(f"  tau_CSTR = {Xdes/(k_op*(1-Xdes)):.5f} s  ({Xdes/(k_op*(1-Xdes))*1000:.2f} ms)")
print(f"  Razon CSTR/PFR = {(Xdes/(k_op*(1-Xdes)))/(-np.log(1-Xdes)/k_op):.3f}x")


---
## Paso 2 — Reactor Batch: tiempo de reacción

### Ecuación de diseño

$$t_{rxn} = C_{A0}\int_0^{X_{des}}\frac{dX}{k\,C_{A0}(1-X)}
= \frac{1}{k}\int_0^{X_{des}}\frac{dX}{1-X} = \frac{-\ln(1-X_{des})}{k}$$

### Equivalencia con el PFR

El tiempo de reacción en el Batch **es idéntico** al tiempo espacial en el PFR.
Ambos aprovechan los gradientes de concentración (alta conversión al inicio → velocidad alta).

### Análisis de escala industrial para Batch

Un reactor Batch de craqueo de etano **no existe industrialmente** porque:
1. Los tiempos de reacción son de milisegundos (imposible cargar/descargar a esa velocidad).
2. Las temperaturas (800°C) requieren hornos continuos, no reactores discontinuos.
3. Los caudales industriales son de miles de ton/año (el Batch no escala económicamente).

Sin embargo, el **análisis del Batch es válido como herramienta de diseño** para
entender la cinética a escala de laboratorio y determinar $k$ experimentalmente.


In [ ]:
# Paso 2: Reactor Batch — craqueo de etano

# Solución analitica (1er orden)
t_rxn_anal = -np.log(1-Xdes)/k_op   # s

# Integracion numerica (verificacion)
mask_b = X_arr <= Xdes
t_rxn_num = np.trapz(CA0/rA_arr[mask_b], X_arr[mask_b])

print("=" * 58)
print("  REACTOR BATCH — Craqueo termico de etano")
print("=" * 58)
print(f"  Solucion analitica: t_rxn = -ln(1-X)/k")
print(f"  t_rxn = -ln(1-{Xdes})/{k_op:.5f}")
print(f"  t_rxn = {t_rxn_anal:.5f} s  =  {t_rxn_anal*1000:.2f} ms")
print()
print(f"  Verificacion numerica: t_rxn = {t_rxn_num:.5f} s")
print(f"  Diferencia analitica-numerica: {abs(t_rxn_anal-t_rxn_num)*1000:.4f} ms")
print()
print(f"  INTERPRETACION INDUSTRIAL:")
print(f"  - Tiempo de reaccion en BATCH: {t_rxn_anal*1000:.2f} milisegundos")
print(f"  - Imposible operar batch a esta escala de tiempo")
print(f"  - El PFR tubular es el unico reactor viable")
print("=" * 58)

# Curva X(t) para batch
t_sim = np.linspace(0, t_rxn_anal*2.5, 500)
X_t   = 1 - np.exp(-k_op*t_sim)

fig, axes = plt.subplots(1, 3, figsize=(17, 5.5))

# X vs t (batch)
ax = axes[0]
ax.plot(t_sim*1000, X_t, '#1565C0', lw=3)
ax.axhline(Xdes, color='#C62828', ls='--', lw=2.2,
           label=f'X_des = {Xdes}')
ax.axvline(t_rxn_anal*1000, color='#2E7D32', ls='--', lw=2.2,
           label=f't_rxn = {t_rxn_anal*1000:.2f} ms')
ax.scatter([t_rxn_anal*1000],[Xdes], s=220, color='gold', zorder=9,
           edgecolors='black', lw=2, marker='*')
ax.set_xlabel('Tiempo t [ms]')
ax.set_ylabel('Conversion X_etano')
ax.set_title('Conversion vs Tiempo — Batch\n'
             '(equivalente al perfil PFR)')
ax.legend(fontsize=9); ax.set_ylim(0, 1.0)

# Levenspiel batch
ax2 = axes[1]
ax2.plot(X_arr, CA0/rA_arr*1000, '#1565C0', lw=3,
         label='C_A0/(-r_A) [ms]')
ax2.fill_between(X_arr[mask_b], 0, (CA0/rA_arr[mask_b])*1000,
                 alpha=0.30, color='#1565C0',
                 label=f't_rxn = {t_rxn_anal*1000:.2f} ms = area')
ax2.axvline(Xdes, color='#C62828', ls='--', lw=2)
ax2.set_xlabel('Conversion X_etano')
ax2.set_ylabel('C_A0/(-r_A) [ms]')
ax2.set_title('Levenspiel — Batch\n(area = tiempo de reaccion en ms)')
ax2.legend(fontsize=9)
ax2.set_ylim(0, (CA0/rA_arr[mask_b]).max()*1000*3.0)

# Sensibilidad a la temperatura en el Batch
T_rng = np.linspace(730, 880, 200)
k_rng = k0*np.exp(-Ea/(R_gas*T_rng))
t_rxn_rng = -np.log(1-Xdes)/k_rng * 1000  # ms

ax3 = axes[2]
ax3.semilogy(T_rng-273.15, t_rxn_rng, '#C62828', lw=3)
ax3.axvline(T_op-273.15, color='#1565C0', ls='--', lw=2.5,
            label=f'T_op={T_op-273.15:.0f}C: t={t_rxn_anal*1000:.2f}ms')
ax3.scatter([T_op-273.15],[t_rxn_anal*1000], s=200, color='gold',
            zorder=9, edgecolors='black', lw=1.5, marker='*')
ax3.set_xlabel('Temperatura [C]')
ax3.set_ylabel('t_rxn [ms] (escala log)')
ax3.set_title(f'Sensibilidad t_rxn vs T\n'
              f'(X_des={Xdes}, Ea={Ea}kJ/mol)')
ax3.legend(fontsize=9)

plt.suptitle('Caso 2 — Paso 2: Reactor Batch — Craqueo de Etano',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


---
## Paso 3 — Reactor CSTR: volumen y análisis de inviabilidad

### Ecuación de diseño del CSTR

$$V_{CSTR} = \frac{F_{A0}\,X_{des}}{k\,C_{A0}(1-X_{des})} = \frac{v_0\,X_{des}}{k\,(1-X_{des})}$$

### ¿Por qué el CSTR es inviable para el craqueo de etano?

El CSTR opera a la concentración de **salida** — la mínima. Para el craqueo de etano
con $X = 0.60$, opera a $C_A = 0.4\,C_{A0}$, es decir, con 40% del etano restante.
Esto hace que la velocidad sea 60% menor que en la entrada del PFR.

Adicionalmente, el CSTR requiere temperatura uniforme de **800°C** en todo el volumen.
Esto implica:
- Materiales de construcción de $\Inconel$ o $\Hastelloy$ — extremadamente costosos.
- Sistema de calefacción uniforme de un tanque agitado a 800°C — prácticamente imposible.
- Riesgo catastrófico de descomposición de etileno a temperaturas tan altas.

El resultado es que el CSTR es **técnicamente inviable** para este proceso.


In [ ]:
# Paso 3: Reactor CSTR — craqueo de etano

tau_CSTR = Xdes/(k_op*(1-Xdes))
V_CSTR   = tau_CSTR * v0     # dm^3

tau_PFR  = -np.log(1-Xdes)/k_op
V_PFR_calc = tau_PFR * v0   # dm^3

print("=" * 58)
print("  REACTOR CSTR — Craqueo termico de etano")
print("=" * 58)
print(f"  tau_CSTR = X_des/(k*(1-X_des))")
print(f"  tau_CSTR = {Xdes}/({k_op:.5f}*{1-Xdes})")
print(f"  tau_CSTR = {tau_CSTR:.5f} s  ({tau_CSTR*1000:.2f} ms)")
print()
print(f"  V_CSTR   = tau*v0 = {tau_CSTR:.5f}*{v0:.0f}")
print(f"  V_CSTR   = {V_CSTR:.3f} dm3  ({V_CSTR/1000:.5f} m3)")
print()
print(f"  COMPARACION DIRECTA:")
print(f"  tau_CSTR = {tau_CSTR*1000:.3f} ms")
print(f"  tau_PFR  = {tau_PFR*1000:.3f} ms")
print(f"  Razon    = {tau_CSTR/tau_PFR:.4f}x")
print(f"  V_CSTR   = {V_CSTR:.3f} dm3")
print(f"  V_PFR    = {V_PFR_calc:.3f} dm3")
print()
print(f"  DIAGNOSTICO INDUSTRIAL:")
print(f"  - CSTR a 800°C: imposible mecánica y termicamente")
print(f"  - Solo el PFR tubular es viable para este proceso")
print("=" * 58)

# Figura comparativa
fig, axes = plt.subplots(1, 3, figsize=(17, 5.5))

# Panel 1: Levenspiel CSTR vs PFR
ax = axes[0]
ax.plot(X_arr, FA0/rA_arr, '#1565C0', lw=3, label='F_A0/(-r_A) [dm3]')
mask_x = X_arr<=Xdes
V_PFR_area = np.trapz((FA0/rA_arr)[mask_x], X_arr[mask_x])
ax.fill_between(X_arr[mask_x], 0, (FA0/rA_arr)[mask_x],
                alpha=0.28, color='#2E7D32',
                label=f'PFR: V={V_PFR_area:.2f} dm3')
Hcstr = FA0/rA_des
ax.fill_between([0,Xdes],[0,0],[Hcstr,Hcstr],
                alpha=0.25, color='#C62828',
                label=f'CSTR: V={V_CSTR:.2f} dm3')
ax.plot([0,Xdes,Xdes],[Hcstr,Hcstr,0],'#C62828',ls='--',lw=2)
ax.set_xlabel('Conversion X_etano')
ax.set_ylabel('F_A0/(-r_A) [dm3]')
ax.set_title('Levenspiel: CSTR vs PFR\n(craqueo termico etano)')
ax.legend(fontsize=9); ax.set_ylim(0, Hcstr*3.5)

# Panel 2: Razon V_CSTR/V_PFR vs X
X_c2 = np.linspace(0.01, 0.99, 300)
tau_P2 = -np.log(1-X_c2)/k_op
tau_C2 = X_c2/(k_op*(1-X_c2))
rat2   = tau_C2/tau_P2

ax2 = axes[1]
ax2.plot(X_c2, rat2, '#1565C0', lw=3)
ax2.axhline(1.0, color='gray', lw=1.2)
ax2.axvline(Xdes, color='#C62828', ls='--', lw=2.2,
            label=f'X_des={Xdes}: razon={np.interp(Xdes,X_c2,rat2):.3f}x')
ax2.fill_between(X_c2, 1, rat2, alpha=0.14, color='#C62828',
                 label='Exceso CSTR')
# Anotaciones de ingenieria
for xv, col in zip([0.3,0.6,0.8,0.9],
                   ['#1976D2','#C62828','#2E7D32','#7B1FA2']):
    rv = np.interp(xv, X_c2, rat2)
    ax2.text(xv+0.01, rv+0.05, f'{rv:.2f}x', fontsize=9,
             color=col, fontweight='bold')
ax2.set_xlabel('Conversion X_etano')
ax2.set_ylabel('V_CSTR / V_PFR')
ax2.set_title('Penalizacion del CSTR\n(1er orden irreversible)')
ax2.legend(fontsize=9)

# Panel 3: Barras de comparison
nombres3 = ['Batch\nt_rxn [ms]','CSTR\n[dm3]','PFR\n[dm3]']
valores3  = [t_rxn_anal*1000, V_CSTR, V_PFR_calc]
colores3  = ['#1565C0','#C62828','#2E7D32']
ax3 = axes[2]
bars3 = ax3.bar(nombres3, valores3, color=colores3,
                alpha=0.85, edgecolor='black', lw=1.5)
for bar, val, uni in zip(bars3, valores3,['ms','dm3','dm3']):
    ax3.text(bar.get_x()+bar.get_width()/2,
             bar.get_height()+max(valores3)*0.01,
             f'{val:.3f} {uni}',
             ha='center', fontsize=9.5, fontweight='bold')
ax3.set_title(f'Comparacion de disenos\nX_des={Xdes}, T={T_op-273.15:.0f}C')

plt.suptitle('Caso 2 — Paso 3: CSTR y comparacion inicial',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


---
## Paso 4 — Reactor PFR: diseño detallado y análisis de sensibilidad

### Ecuación de diseño del PFR (1er orden, solución analítica)

$$V_{PFR} = \frac{F_{A0}}{k}\int_0^{X_{des}}\frac{dX}{C_{A0}(1-X)/C_{A0}}
= \frac{F_{A0}}{k\,C_{A0}}\,[-\ln(1-X_{des})] = \frac{v_0\,(-\ln(1-X_{des}))}{k}$$

Simplificando con $\tau_{PFR} = -\ln(1-X_{des})/k$:

$$V_{PFR} = v_0\,\tau_{PFR}$$

### Diseño industrial real: los hornos de craqueo

Los hornos de steam cracking industriales son PFRs de tubos de alta temperatura:

| Parámetro | Valor típico |
|-----------|-------------|
| Diámetro tubo | 80-120 mm |
| Longitud tubo | 40-80 m |
| Temperatura pared | 900-1100°C |
| Tiempo de residencia | 0.1-0.5 s |
| Producción por horno | 100-300 kt etileno/año |

### Verificación de diseño

Para el caudal y la conversión dados, el volumen calculado corresponde a
un conjunto de tubos en paralelo que conforman el horno de craqueo.


In [ ]:
# Paso 4: Reactor PFR — craqueo termico de etano (análisis completo)

# Calculo del PFR
tau_PFR_op = -np.log(1-Xdes)/k_op    # s
V_PFR_op   = tau_PFR_op * v0          # dm^3
V_PFR_m3   = V_PFR_op / 1000          # m^3

# Perfil de conversion a lo largo del PFR
V_span = np.linspace(0, V_PFR_op*1.5, 1000)
X_PFR_profile = 1 - np.exp(-k_op * V_span/v0)

# Perfil de velocidad de reaccion
rA_PFR_profile = k_op * CA0 * (1 - X_PFR_profile)

print("=" * 58)
print("  REACTOR PFR — Craqueo termico de etano")
print("=" * 58)
print(f"  Conversion deseada X_des   = {Xdes}")
print(f"  k(T_op)                    = {k_op:.5f} s^-1")
print(f"  tau_PFR = -ln(1-X)/k       = {tau_PFR_op:.5f} s ({tau_PFR_op*1000:.3f} ms)")
print(f"  V_PFR   = tau_PFR * v0     = {V_PFR_op:.3f} dm3")
print(f"  V_PFR                      = {V_PFR_m3:.5f} m3")
print()
print(f"  DATOS DE DISEÑO TUBULAR:")
# Dimensionar como tubo
D_tubo = 0.10       # m  (diametro tipico)
A_tubo = np.pi*(D_tubo/2)**2  # m^2
L_tubo = V_PFR_m3 / A_tubo   # m

print(f"  Si D_tubo = {D_tubo*100:.0f} cm:")
print(f"  L_tubo = V_PFR/A = {L_tubo:.2f} m")
if L_tubo > 80:
    n_tubos = int(np.ceil(L_tubo/60))
    print(f"  -> Se necesitan {n_tubos} tubos en paralelo de 60m")
    print(f"  -> Arreglo tipico de horno de craqueo industrial")
else:
    print(f"  -> Un unico tubo de {L_tubo:.2f} m (factible)")
print("=" * 58)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Caso 2 — Paso 4: Reactor PFR — Craqueo Termico de Etano\n'
             f'T={T_op-273.15:.0f}C, X_des={Xdes}, tau={tau_PFR_op*1000:.2f}ms',
             fontsize=12, fontweight='bold', y=1.01)

# X(V)
ax = axes[0,0]
ax.plot(V_span, X_PFR_profile, '#2E7D32', lw=3)
ax.axhline(Xdes, color='#C62828', ls='--', lw=2.2,
           label=f'X_des = {Xdes}')
ax.axvline(V_PFR_op, color='#2E7D32', ls='--', lw=2.2,
           label=f'V_PFR = {V_PFR_op:.2f} dm3')
ax.scatter([V_PFR_op],[Xdes], s=220, color='gold', zorder=9,
           edgecolors='black', lw=2, marker='*')
ax.set_xlabel('Volumen V [dm3]')
ax.set_ylabel('Conversion X_etano')
ax.set_title('Perfil X(V) en el PFR')
ax.legend(fontsize=9)

# -r_A(V)
ax2 = axes[0,1]
ax2.plot(V_span, rA_PFR_profile*1000, '#C62828', lw=3)
ax2.axvline(V_PFR_op, color='#2E7D32', ls='--', lw=2.2,
            label=f'V_PFR = {V_PFR_op:.2f} dm3')
ax2.set_xlabel('Volumen V [dm3]')
ax2.set_ylabel('-r_A [mmol/(dm3*s)]')
ax2.set_title('Velocidad de reaccion a lo largo del PFR')
ax2.legend(fontsize=9)

# Sensibilidad a T: tau_PFR y V_PFR
T_rng2  = np.linspace(730, 880, 200)
k_rng2  = k0*np.exp(-Ea/(R_gas*T_rng2))
tau_rng = -np.log(1-Xdes)/k_rng2 * 1000  # ms
V_rng   = tau_rng/1000 * v0                # dm^3

ax3 = axes[1,0]
ax3.semilogy(T_rng2-273.15, tau_rng, '#1565C0', lw=3,
             label='tau_PFR [ms]')
ax3_r = ax3.twinx()
ax3_r.semilogy(T_rng2-273.15, V_rng, '#C62828', lw=2.5, ls='--',
               label='V_PFR [dm3]')
ax3.axvline(T_op-273.15, color='gray', ls=':', lw=2)
ax3.set_xlabel('Temperatura [C]')
ax3.set_ylabel('tau_PFR [ms]', color='#1565C0')
ax3_r.set_ylabel('V_PFR [dm3]', color='#C62828')
ax3.set_title('Sensibilidad tau_PFR y V_PFR vs T\n'
              '(log — muy fuerte por Ea=272 kJ/mol)')
lines1, labs1 = ax3.get_legend_handles_labels()
lines2, labs2 = ax3_r.get_legend_handles_labels()
ax3.legend(lines1+lines2, labs1+labs2, fontsize=9)

# Tabla resumen final
ax4 = axes[1,1]; ax4.axis('off')
tabla = [
    ['Reactor', 'Tau / t_rxn', 'Volumen', 'Viabilidad'],
    ['Batch',   f'{tau_PFR_op*1000:.3f} ms', '---',           'Solo laboratorio'],
    ['CSTR',    f'{tau_CSTR*1000:.3f} ms',   f'{V_CSTR:.2f} dm3', '❌ INVIABLE (800°C)'],
    ['PFR',     f'{tau_PFR_op*1000:.3f} ms', f'{V_PFR_op:.2f} dm3','✓ INDUSTRIAL'],
    ['V_CSTR/V_PFR', f'{tau_CSTR/tau_PFR_op:.3f}x', '---', 'Razon volumetrica'],
]
tbl = ax4.table(cellText=tabla[1:], colLabels=tabla[0],
                cellLoc='center', loc='center', bbox=[0,0.2,1,0.75])
tbl.auto_set_font_size(False); tbl.set_fontsize(11)
for j in range(4):
    tbl[0,j].set_facecolor('#b71c1c')
    tbl[0,j].set_text_props(color='white', fontweight='bold')
colores_tab = ['#E8F5E9','#FFCCBC','#C8E6C9','#F5F5F5']
for i in range(1,5):
    for j in range(4): tbl[i,j].set_facecolor(colores_tab[i-1])
ax4.set_title('Tabla comparativa final', fontweight='bold', fontsize=11)

plt.tight_layout(); plt.show()


---
## Paso 5 — Widget interactivo: impacto de T y X en el diseño del PFR

### ¿Cómo opera el ingeniero el horno de craqueo?

En planta real, el operador controla principalmente:
1. **Temperatura de salida del horno** → ajusta la conversión de etano.
2. **Caudal de vapor de agua** → controla la dilución y el tiempo de residencia.
3. **Temperatura de pared del tubo** → limita el coque depositado.

El widget siguiente simula cómo cambia el diseño del PFR al variar
la temperatura de operación y la conversión deseada.


In [ ]:
# Paso 5: Widget interactivo — sensibilidad T y X en el PFR

def diseno_pfr_craqueo(T_C=800., X_des_w=0.60):
    T_K   = T_C + 273.15
    k_w   = k0*np.exp(-Ea/(R_gas*T_K))
    tau_w = -np.log(1-X_des_w)/k_w
    V_w   = tau_w*v0

    X_rng = np.linspace(0.001, 0.99, 400)
    rA_w  = k_w*CA0*(1-X_rng)
    tau_C_w = X_rng/(k_w*(1-X_rng))
    tau_P_w = -np.log(1-X_rng)/k_w

    fig, axes = plt.subplots(1, 3, figsize=(17, 5.5))
    fig.suptitle(f'Diseno PFR de Craqueo Termico de Etano\n'
                 f'T={T_C:.0f}C, k={k_w:.4f}s-1, X_des={X_des_w:.2f}',
                 fontsize=12, fontweight='bold', y=1.01)

    # Levenspiel
    ax = axes[0]
    Lev_w = FA0/rA_w
    mask_w = X_rng<=X_des_w
    ax.plot(X_rng, Lev_w, '#1565C0', lw=3)
    ax.fill_between(X_rng[mask_w], 0, Lev_w[mask_w],
                    alpha=0.28, color='#2E7D32',
                    label=f'PFR: tau={tau_w*1000:.2f}ms, V={V_w:.2f}dm3')
    Hc_w = FA0/(k_w*CA0*(1-X_des_w))
    ax.fill_between([0,X_des_w],[0,0],[Hc_w,Hc_w],
                    alpha=0.22, color='#C62828',
                    label=f'CSTR: tau={X_des_w/(k_w*(1-X_des_w))*1000:.2f}ms')
    ax.plot([0,X_des_w,X_des_w],[Hc_w,Hc_w,0],'#C62828',ls='--',lw=2)
    ax.set_xlabel('Conversion X_etano')
    ax.set_ylabel('F_A0/(-r_A) [dm3]')
    ax.set_title('Levenspiel')
    ax.legend(fontsize=9); ax.set_ylim(0, Hc_w*4.0)

    # X(V) del PFR
    ax2 = axes[1]
    V_sp = np.linspace(0, V_w*1.5, 500)
    X_prf = 1 - np.exp(-k_w*V_sp/v0)
    ax2.plot(V_sp, X_prf, '#2E7D32', lw=3)
    ax2.axhline(X_des_w, color='#C62828', ls='--', lw=2.2)
    ax2.axvline(V_w, color='#2E7D32', ls='--', lw=2.2,
                label=f'V={V_w:.3f}dm3 ({V_w/1000:.6f}m3)')
    ax2.scatter([V_w],[X_des_w], s=220, color='gold', zorder=9,
                edgecolors='black', lw=2, marker='*')
    ax2.set_xlabel('V [dm3]'); ax2.set_ylabel('X_etano')
    ax2.set_title('Perfil de conversion en PFR')
    ax2.legend(fontsize=9)

    # tau_PFR vs X para esta temperatura
    ax3 = axes[2]
    ax3.plot(X_rng, tau_P_w*1000, '#2E7D32', lw=3, label='tau PFR [ms]')
    ax3.plot(X_rng, tau_C_w*1000, '#C62828', lw=3, label='tau CSTR [ms]')
    ax3.axvline(X_des_w, color='#1565C0', ls='--', lw=2.2,
                label=f'X_des={X_des_w:.2f}')
    ax3.set_xlabel('Conversion X_etano')
    ax3.set_ylabel('Tiempo espacial [ms]')
    ax3.set_title('tau_PFR vs tau_CSTR')
    ax3.legend(fontsize=9); ax3.set_ylim(0, tau_C_w[X_rng<0.95].max()*1000*1.2)

    plt.tight_layout(); plt.show()
    razon_w = X_des_w/(k_w*(1-X_des_w))/(-np.log(1-X_des_w)/k_w)
    print(f"  T={T_C:.0f}C | k={k_w:.4f}s-1 | X_des={X_des_w:.2f}")
    print(f"  tau_PFR = {tau_w*1000:.3f} ms | V_PFR = {V_w:.4f} dm3")
    print(f"  tau_CSTR= {X_des_w/(k_w*(1-X_des_w))*1000:.3f} ms")
    print(f"  Razon CSTR/PFR = {razon_w:.4f}x")

interact(diseno_pfr_craqueo,
    T_C    =FloatSlider(value=800., min=700., max=900., step=5.,
                        description='T [C]',
                        style={'description_width':'100px'}),
    X_des_w=FloatSlider(value=0.60, min=0.10, max=0.90, step=0.02,
                        description='X deseada',
                        style={'description_width':'100px'})
);


---
## Conclusiones del Caso 2

### Resultados para T = 800°C, X_des = 0.60

| Reactor | Tiempo espacial | Volumen | Viabilidad |
|---------|----------------|---------|------------|
| **Batch** | $t_{rxn}$ ms | — | Solo I+D, no producción |
| **CSTR** | mayor | mayor | **Inviable** (800°C en tanque agitado) |
| **PFR** | menor | menor | **Industrial** (hornos tubulares) |

### Lecciones clave de este caso

1. **La energía de activación alta** ($E_a = 272$ kJ/mol) hace que la reacción
   sea extremadamente sensible a la temperatura. Cada 10°C cambia $k$ en ~2.5×.

2. **El CSTR es doblemente inviable**: mecánicamente (no se puede hacer un tanque
   agitado a 800°C con escala industrial) y cinéticamente (mayor volumen que el PFR).

3. **El PFR es el único reactor viable** para el steam cracking. La industria usa
   hornos de tubos en paralelo con tiempo de residencia de 0.1-0.5 s.

4. **El tiempo de residencia corto** (milisegundos) es clave para evitar reacciones
   secundarias de etileno → acetileno, benceno, coque.

5. **El Batch tiene valor académico y de I+D**: permite medir la cinética en
   microreactores a distintas temperaturas para determinar $k_0$ y $E_a$.

---

### Comparación final entre los dos casos

| Aspecto | Caso 1 (Estireno) | Caso 2 (Etileno) |
|---------|-------------------|-----------------|
| Tipo de reacción | Reversible, endotérmica | Irreversible, endotérmica |
| Temperatura | 600°C | 800°C |
| Tiempo de residencia | Minutos | Milisegundos |
| Limitación | Equilibrio termodinámico | Reacciones secundarias |
| Reactor industrial | PFR adiabático + reheaters | Hornos tubulares (PFR) |
| Factor crítico | $X_{des}$ vs $X_{eq}$ | Control de temperatura |
| Rol del CSTR | Posible pero subóptimo | Inviable técnicamente |

---
*Caso 2 — Programa 740484 · Maestría Ingeniería Química · Universidad del Valle · 2024*
